In [1]:
import numpy as np
from helpers import *
from implementations import *

Import the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [44]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [45]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)

(328135, 321)
(109379, 321)
(328135,)


Convert -1 into 0 in y

In [46]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


In [47]:
x_copy = x_train.copy()
x_copy.shape

(328135, 321)

In [48]:
def remove_irrelevant_geographic_features(data, column_names):
    """
    Removes specific geographic-related features from the dataset using NumPy.

    Parameters:
    data (np.ndarray): The input NumPy array containing BRFSS data.
    column_names (list of str): List of all column names in the dataset.

    Returns:
    np.ndarray: A NumPy array with irrelevant geographic features removed.
    """
    # List of columns to drop based on Category 1
    columns_to_remove = ['_STATE', '_LLCPWT', '_PSU', '_STSTR'] #GEOGRAPHIC
    
    # Find the indices of columns to remove
    indices_to_remove = [column_names.index(col) for col in columns_to_remove if col in column_names]
    
    # Remove the specified columns
    data_filtered = np.delete(data, indices_to_remove, axis=1)
    
    return data_filtered

# Example usage
# Assume `data` is the NumPy array with BRFSS data and `column_names` is a list of column names
# data_filtered = remove_irrelevant_geographic_features(data, column_names)



In [50]:
x_train_filtered = remove_irrelevant_geographic_features(x_copy, headers_train)
x_train_filtered.shape

(328135, 310)

Remove columns with Nan

In [51]:
# Function to find the count of missing values in each columns
def find_missing_values(data, headers=None):
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

In [52]:
# Function to remove columns with more than 1/3 of missing values
def remove_high_missing_columns(data):
    num_rows, num_cols = data.shape
    threshold = 0
    missing_count = find_missing_values(data) 

    # Create a mask for columns to keep
    columns_to_keep = [col for col in range(num_cols) if missing_count.get(col, 0) <= threshold]

    # Return a new array with only the columns that meet the criteria
    return data[:, columns_to_keep]

In [53]:
sliced_x_train = remove_high_missing_columns(x_train_filtered)

In [54]:
sliced_x_train.shape

(328135, 79)

In [55]:
column_with_nan = find_missing_values(sliced_x_train)

In [56]:
print(column_with_nan)

{}


Standardize data to avoid issues with large numbers

In [57]:
def standardize(x):
    """Stadartize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standartized data, shape=(num_samples, num_features)

    >>> standardize(np.array([[1, 2], [3, 4], [5, 6]]))
    array([[-1.22474487, -1.22474487],
           [ 0.        ,  0.        ],
           [ 1.22474487,  1.22474487]])
    """
    # ***************************************************
    means=np.mean(x, axis=0)
    stds=np.std(x, axis=0)
    std_data=(x-means)/stds
    # ***************************************************

    return std_data

In [58]:
x_train = standardize(sliced_x_train)

Split the data

In [59]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.

    >>> split_data(np.arange(13), np.arange(13), 0.8, 1)
    (array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]), array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]))
    """
    # set seed
    np.random.seed(seed)
    # ***************************************************
    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te
    # ***************************************************

In [60]:
x_tr, x_val, y_tr, y_val = split_data(x_train, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 79)
(65627, 79)
(262508,)
(65627,)


Train the model

In [61]:
initial_w = np.zeros(x_tr.shape[1])
max_iters = 1000
gamma = 0.01

w_opt, loss_opt = logistic_regression(y_tr, x_tr, initial_w, max_iters, gamma)

Current iteration=0, loss=0.6931471805599448
Current iteration=100, loss=0.6810706074728802
Current iteration=200, loss=0.6787047044245288
Current iteration=300, loss=0.6777112838691467
Current iteration=400, loss=0.6771756157784183
Current iteration=500, loss=0.6768557051417394
Current iteration=600, loss=0.6766525631966053
Current iteration=700, loss=0.6765175682496793
Current iteration=800, loss=0.6764242251205262
Current iteration=900, loss=0.6763571332306489
loss=0.6763069721601513


In [62]:
print(w_opt)

[ 0.00233518  0.00047801  0.00059428 -0.00489284 -0.00571038 -0.00153995
 -0.00201609 -0.00201609 -0.00925499  0.00013607 -0.00915797 -0.08196477
 -0.0164814  -0.01865783 -0.05631392 -0.03145909 -0.03823772 -0.10540619
 -0.0047917  -0.0235687   0.02122858  0.08965453 -0.00110382  0.0023401
 -0.00281546  0.00095805  0.01666365 -0.00185252  0.0331001   0.10159679
 -0.01521301  0.02384493  0.01968776 -0.0214987  -0.00406353 -0.0024641
  0.00215762  0.00135417  0.00085311  0.00067777  0.05318358  0.02900685
  0.05599326  0.02194268 -0.00196539  0.0008049  -0.01689028 -0.03175907
 -0.03894219  0.01474321  0.03237053 -0.01257202 -0.00653817 -0.00060121
 -0.00396887 -0.00272533 -0.00438797  0.00100613  0.00086089  0.00479341
  0.00379235 -0.00483742  0.00436134 -0.00052833 -0.00115701  0.00763054
 -0.02187237 -0.01639419 -0.00996842  0.01747015 -0.0064725   0.00785059
  0.00404206 -0.01040859 -0.00147146  0.01659111 -0.0083155   0.00076696
  0.00231158]


Apply the model to the validation set

In [63]:
g = x_val @ w_opt
g.shape

(65627,)

In [64]:
limit = 0.5

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


Check our accuracy and F1

In [65]:
accuracy = np.mean(y_val == g)

print("Accuracy:", accuracy)

Accuracy: 0.8834321239733646


In [66]:
TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

Precision: 0.33299714237687006
Recall: 0.34981458590852904
F1 Score: 0.34119875990354803
